# Example 12.8 / 1: Load Predictions and Backtest the Plan

Companion notebook for **12.8 Machine Learning and Optimization** in the AMPL lab Documentation, section "Load Predictions and Backtest the Plan". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Load Predictions and Backtest the Plan

Align predictions to the model's keys before assigning them, and check units, signs and missing values before solving.

The steel products receive demand predictions from a trend fitted on four past periods: the
last value plus the average step, so bands `6200 + 400 = 6600` and coils `3300 + 100 = 3400`
tons. The model caps production at the prediction and adds the hours limit of the steel model from Model Components (AMPL lab documentation, section 1).
data.py builds `prediction_by_product` with `zip(..., strict=True)` and rejects keys that differ
from `PRODUCTS`; run.py freezes the plan read with `to_dict()`, scores it on realized demand, and
re-solves with that demand as the hindsight benchmark.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PRODUCTS;                           # products to plan
param predicted_demand {PRODUCTS} >= 0; # tons, from the predictor
param unit_profit {PRODUCTS};           # dollars per ton
param rate {PRODUCTS} > 0;              # tons per hour
param avail >= 0;                       # hours

var Make {p in PRODUCTS} >= 0 <= predicted_demand[p];  # tons; capped by the prediction

maximize PredictedProfit:               # profit under predicted demand, dollars
    sum {p in PRODUCTS} unit_profit[p] * Make[p];

subject to Time:                        # production hours within availability
    sum {p in PRODUCTS} Make[p] / rate[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
# A trend predictor fitted on four past periods per product.
history = {"bands": [5000, 5400, 5800, 6200], "coils": [3000, 3100, 3200, 3300]}
product_ids = list(history)
predicted_values = [h[-1] + (h[-1] - h[0]) / (len(h) - 1) for h in history.values()]

ampl.set["PRODUCTS"] = ["bands", "coils"]
ampl.param["unit_profit"] = {"bands": 25, "coils": 30}
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["avail"] = 40

prediction_by_product = dict(zip(product_ids, predicted_values, strict=True))
if set(prediction_by_product) != set(ampl.set["PRODUCTS"].to_list()):
    raise ValueError("prediction keys do not match PRODUCTS")
ampl.param["predicted_demand"] = prediction_by_product

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
plan = ampl.var["Make"].to_dict()
print(ampl.solve_result, prediction_by_product, plan, ampl.obj["PredictedProfit"].value())

# Backtest: realized demand arrives after the plan is frozen.
realized = {"bands": 6000, "coils": 3800}
profit = ampl.param["unit_profit"].to_dict()
realized_profit = sum(profit[p] * min(plan[p], realized[p]) for p in plan)

# Hindsight benchmark: re-solve with the realized demand.
ampl.param["predicted_demand"] = realized
ampl.solve(solver="highs", verbose=False)
hindsight = ampl.obj["PredictedProfit"].value()
print(realized_profit, hindsight, hindsight - realized_profit)

solved {'bands': 6600.0, 'coils': 3400.0} {'bands': 6600, 'coils': 980} 194400.0
179400 192000.0 12600.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`; the predictions arrive as data, and `let`
replaces them with the realized demand for the hindsight solve:

In [6]:
%%writefile data.dat
set PRODUCTS := bands coils;
param:   predicted_demand  unit_profit  rate :=
  bands              6600           25   200
  coils              3400           30   140;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Make, PredictedProfit;
let predicted_demand['bands'] := 6000;
let predicted_demand['coils'] := 3800;
solve;
display PredictedProfit;

HiGHS 1.14.0: optimal solution; objective 194400
0 simplex iterations
0 barrier iterations
Make [*] :=
bands  6600
coils   980
;

PredictedProfit = 194400

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
PredictedProfit = 192000

